In [1]:
import sys
from pathlib import Path

SRC_DIR = Path.cwd().parent / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
import traceback

try:
    import vista.clinvar
    print("Imported successfully")
except Exception:
    traceback.print_exc()

Imported successfully


In [3]:
import importlib
import vista.clinvar

importlib.reload(vista.clinvar)

print(vista.clinvar.__dict__.keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__file__', '__cached__', '__builtins__', 'time', 'requests', 'Variant', 'ClinVarClient'])


In [4]:
from vista.intake import create_variant
from vista.clinvar import ClinVarClient

In [5]:
variant = create_variant(
    gene="PLIN4",
    transcript="NM_001367868.2",
    hgvs="c.3702+5G>A",
    classification="VUS",
    zygosity="Heterozygous",
    phenotype="Myopathy",
    inheritance="Autosomal dominant",
)

In [6]:
client = ClinVarClient()

variant = client.annotate(variant)

variant

Variant(gene='PLIN4', hgvs='c.3702+5G>A', transcript='NM_001367868.2', classification='VUS', zygosity='Heterozygous', phenotype='Myopathy', inheritance='Autosomal dominant', chromosome=None, position=None, ref=None, alt=None, genome_build=None, rsid=None, most_severe_consequence=None, clinvar={'found': False, 'query': 'PLIN4[gene] AND c.3702+5G>A', 'records': []}, gnomad={}, predictions={}, pubmed=[], acmg={}, report={})

In [7]:
from vista.intake import create_variant
from vista.clinvar import ClinVarClient

variant = create_variant(
    gene="PLIN4",
    transcript="NM_001367868.2",
    hgvs="c.3702+5G>A",
    classification="VUS",
    zygosity="Heterozygous",
    phenotype="Myopathy",
    inheritance="Autosomal dominant",
)

client = ClinVarClient()

print(client.build_query(variant))

PLIN4[gene] AND c.3702+5G>A


In [8]:
import requests

print(requests.__version__)

2.34.2


In [9]:
query = client.build_query(variant)

response = client.search(query)

type(response)

dict

In [10]:
response.keys()

dict_keys(['header', 'esearchresult'])

In [11]:
import json

print(json.dumps(response["esearchresult"], indent=2))

{
  "count": "0",
  "retmax": "0",
  "retstart": "0",
  "idlist": [],
  "translationset": [],
  "translationstack": [
    {
      "term": "PLIN4[gene]",
      "field": "gene",
      "count": "511",
      "explode": "N"
    },
    {
      "term": "c0x2e37020x2b5G0x3eA[All Fields]",
      "field": "All Fields",
      "count": "4",
      "explode": "N"
    },
    "AND"
  ],
  "querytranslation": "PLIN4[gene] AND c0x2e37020x2b5G0x3eA[All Fields]"
}


In [12]:
print(client.search("PLIN4")["esearchresult"]["count"])

511


In [13]:
print(client.search("c.3702+5G>A")["esearchresult"]["count"])

4


In [14]:
print(client.search('"c.3702+5G>A"')["esearchresult"]["count"])

4


In [15]:
response = client.search("c.3702+5G>A")

response["esearchresult"]["idlist"]

['3614854', '1677531', '1344476', '1043646']

In [16]:
for cid in response["esearchresult"]["idlist"]:
    summary = client.fetch_summary(cid)

    print("=" * 60)
    print("ClinVar ID:", cid)
    print(summary)

ClinVar ID: 3614854
{'header': {'type': 'esummary', 'version': '0.3'}, 'result': {'uids': ['3614854'], '3614854': {'uid': '3614854', 'obj_type': 'single nucleotide variant', 'accession': 'VCV003614854', 'accession_version': 'VCV003614854.2', 'title': 'NM_004006.3(DMD):c.4071+5G>A', 'variation_set': [{'measure_id': '3742373', 'variation_xrefs': [], 'variation_name': 'NM_004006.3(DMD):c.4071+5G>A', 'cdna_change': 'NM_004006.3(DMD):c.4071+5G>A', 'aliases': [], 'variation_loc': [{'status': 'current', 'assembly_name': 'GRCh38', 'chr': 'X', 'band': 'Xp21.1', 'start': '32438236', 'stop': '32438236', 'inner_start': '', 'inner_stop': '', 'outer_start': '', 'outer_stop': '', 'display_start': '32438236', 'display_stop': '32438236', 'assembly_acc_ver': 'GCF_000001405.38', 'annotation_release': '', 'alt': '', 'ref': ''}, {'status': 'previous', 'assembly_name': 'GRCh37', 'chr': 'X', 'band': 'Xp21.1', 'start': '32456353', 'stop': '32456353', 'inner_start': '', 'inner_stop': '', 'outer_start': '', 'ou

In [17]:
from vista.clinvar import ClinVarClient

print(dir(ClinVarClient))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', 'annotate', 'build_query', 'fetch_summary', 'parse_summary', 'search']


In [18]:
query = "NM_001367868.2:c.3702+5G>A"

response = client.search(query)

print(response["esearchresult"]["count"])
print(response["esearchresult"]["idlist"])

0
[]


In [19]:
query = "NM_001367868.2"

response = client.search(query)

print(response["esearchresult"]["count"])

368


In [20]:
query = client.build_query(variant)

response = client.search(query)

ids = response["esearchresult"]["idlist"]

print(ids)

[]


In [21]:
summary = client.fetch_summary("3614854")

parsed = client.parse_summary(summary)

parsed

{'clinvar_id': '3614854',
 'accession': 'VCV003614854',
 'title': 'NM_004006.3(DMD):c.4071+5G>A',
 'classification': 'Uncertain significance',
 'review_status': 'criteria provided, single submitter',
 'last_evaluated': '2024/04/15 00:00',
 'gene': 'DMD'}

In [22]:
variant = client.annotate(variant)

variant.clinvar

{'found': False, 'query': 'PLIN4[gene] AND c.3702+5G>A', 'records': []}